## In this project I'm building an AI Chatbot which answers the questions asked from it.
First step is to install dependencies


In [ ]:
!pip install pypdf
!pip install sentence-transformers
!pip install faiss-cpu
!pip install google-generativeai

## Configure Gemini API

In [3]:
!pip install -q google-genai
from google import genai

client = genai.Client(
    api_key="AQ.Ab8RN6JkBPk5nbpsuGsuL8ozTWLiOiuUNc9p7ga_j5pyduMSNA"
)

## Next step is to upload pdf

In [ ]:
from google.colab import files

uploaded = files.upload()

## EXTRACT TEXT

In [34]:
from pypdf import PdfReader

reader = PdfReader("Gusto_Employee_Handbook_Infographic.pdf")

pages = []

for page_num, page in enumerate(reader.pages, start=1):

    pages.append(
        {
            "page": page_num,
            "text": page.extract_text()
        }
    )

## CHUNK TEXT
Breaking large document into smaller managable segments.

In [37]:
'''def chunk_text(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunks.append(
            text[i:i+chunk_size]
        )

    return chunks

chunks = chunk_text(text)'''

# Doing this with chunks overlapping to preserve data.

def chunk_pages(
    pages,
    chunk_size=500,
    overlap=100
):

    chunks = []

    for page in pages:

        text = page["text"]

        start = 0

        while start < len(text):

            end = start + chunk_size

            chunks.append(
                {
                    "text": text[start:end],
                    "page": page["page"]
                }
            )

            start += chunk_size - overlap

    return chunks

In [39]:
chunks = chunk_pages(pages)

In [ ]:
print(chunks[0])

In [41]:
chunk_texts = [
    chunk["text"]
    for chunk in chunks
]

## Creating embeddings
converting large data (words, sentences) into list of numbers known as vectors.

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

embeddings = embedding_model.encode(
    chunk_texts
)

## Now creating vector database
we're using FAISS

In [12]:
import faiss
import numpy as np

embeddings = np.array(embeddings)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(
    dimension
)

index.add(embeddings)

## Retrieval Function

In [43]:
def retrieve_context(question, k=3):

    question_embedding = embedding_model.encode(
        [question]
    )

    distances, indices = index.search(
        np.array(question_embedding),
        k
    )

    return [
        chunks[i]
        for i in indices[0]
    ]

## Testing the Retrieval Function

In [ ]:
question = "How does Employee Handbook opens up the lines of communication?"

results = retrieve_context(question)

for chunk in results:
    print(chunk)
    print("-" * 50)

## Initializing Gemini Model

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain RAG in one sentence."
)

print(response.text)

## CONNECTING GEMINI

In [68]:
def ask(question,history):

    retrieved_chunks = retrieve_context(
        question
    )

    context = "\n\n".join(
        chunk["text"]
        for chunk in retrieved_chunks
    )

    prompt = f"""
    Use ONLY the context.

    Context:
    {context}

    Question:
    {question}
    """

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    answer = response.text

    pages = sorted(
        set(
            chunk["page"]
            for chunk in retrieved_chunks
        )
    )

    return answer, f'Page {pages}'

##TEST

In [ ]:
ask(
    "What does Employee Handbook do?",'history'
)

## CREATING A FRONTEND USING GRADIO

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

gr.ChatInterface(
    ask
).launch(debug=True)